# 08 - Intégration API et web

Objectif : préparer les modifications de l'API et de Streamlit pour passer par un pipeline central, sans modifier les fichiers finaux maintenant.

Fichiers concernés : `api/main.py`, `app/streamlit_app.py`, `app/gradio_app.py`, futur `src/pipeline.py`.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
SAMPLE_IMAGES_DIR = DATA_DIR / "sample_images"
SRC_DIR = PROJECT_ROOT / "src"
API_DIR = PROJECT_ROOT / "api"
APP_DIR = PROJECT_ROOT / "app"
EVAL_DIR = PROJECT_ROOT / "eval"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
PROMPTS_DIR = PROJECT_ROOT / "prompts"
TESTS_DIR = PROJECT_ROOT / "tests"
OUTPUTS_DIR.mkdir(exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)

In [ ]:
api_code = (API_DIR / "main.py").read_text(encoding="utf-8") if (API_DIR / "main.py").exists() else ""
streamlit_code = (APP_DIR / "streamlit_app.py").read_text(encoding="utf-8") if (APP_DIR / "streamlit_app.py").exists() else ""
gradio_code = (APP_DIR / "gradio_app.py").read_text(encoding="utf-8") if (APP_DIR / "gradio_app.py").exists() else ""
checks = {
    "api_uses_toy_predict": "toy_predict" in api_code,
    "api_uses_run_pipeline": "run_pipeline" in api_code,
    "streamlit_uses_toy_predict": "toy_predict" in streamlit_code,
    "streamlit_uses_run_pipeline": "run_pipeline" in streamlit_code,
    "gradio_optional_uses_toy_predict": "toy_predict" in gradio_code,
}
checks

## Code cible à intégrer dans `api/main.py`

```python
from src.pipeline import run_pipeline

@app.post("/predict")
async def predict(file: UploadFile = File(...)) -> dict:
    UPLOAD_DIR.mkdir(exist_ok=True)
    filename = Path(file.filename or "image.png").name
    suffix = Path(filename).suffix or ".png"
    stem = Path(filename).stem or "image"
    safe_stem = re.sub(r"[^A-Za-z0-9_.-]+", "_", stem)
    target = UPLOAD_DIR / f"uploaded_{safe_stem}{suffix}"
    with target.open("wb") as f:
        shutil.copyfileobj(file.file, f)
    return run_pipeline(target, mode="improved", case_id=safe_stem)
```

## Code cible à intégrer dans `app/streamlit_app.py`

```python
from src.pipeline import run_pipeline

pred = run_pipeline(tmp_path, mode=mode, case_id=Path(uploaded.name).stem)
st.json(pred)
```

Gradio est optionnel. Streamlit est recommandé comme démo officielle.

Conclusion : attendre `src/pipeline.py`, puis remplacer les appels directs à `toy_predict` dans API et Streamlit.